# 环境准备

[docker-compose.yml](./assets/docker-compose.yml) 修改于[github.com/milvus-io/milvus](https://github.com/milvus-io/milvus/blob/master/deployments/docker/standalone/docker-compose.yml)

![](./images/milvus.png)


```sh
# 启动
docker compose up -d
# 关闭
docker compose down
```

完整过程

```sh
pkmer@DESKTOP-2368UCO:~
$ cd software/

pkmer@DESKTOP-2368UCO:software
$ sudo service docker start
[sudo] password for pkmer:123456

pkmer@DESKTOP-2368UCO:software
$ docker compose up -d
[+] up 4/4
 ✔ Network milvus              Created   
 ✔ Container milvus-minio      Started  
 ✔ Container milvus-etcd       Started 
 ✔ Container milvus-standalone Started  
```

下载[attu v3.0.1](https://github.com/zilliztech/attu/releases/tag/v3.0.1)attu-3.0.1-x64.zip 连接成功

![](./images/attn.png)

---

![](./images/attu.png)

# huggingface嵌入向量

[langchain-huggingface](https://reference.langchain.com/python/langchain-huggingface)

基础版 `langchain-huggingface` **不包含** `sentence-transformers` 和 `transformers` 这两个跑本地模型必需的库。

```
uv add langchain-huggingface sentence-transformers torch
```

In [3]:
import os

import dotenv

dotenv.load_dotenv()
print(os.environ["HF_HOME"], os.environ["HF_ENDPOINT"])

D:\huggingface_cache https://hf-mirror.com


In [4]:
import torch

print("CUDA 是否可用:", torch.cuda.is_available())

CUDA 是否可用: False


512 维度！ 这正是 [bge-small-zh-v1.5](https://hf-mirror.com/BAAI/bge-small-zh-v1.5)  只输出稠密向量（Dense）

In [5]:
import time

from langchain_huggingface import HuggingFaceEmbeddings

print("正在加载模型...")
start_time = time.time()
model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")
end_time = time.time()
print(f"模型加载完成，耗时: {end_time - start_time:.2f}秒")

正在加载模型...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

模型加载完成，耗时: 20.00秒


In [6]:
# 测试生成向量
vector = model.embed_query("Milvus 是一个强大的向量数据库。")

In [7]:
print(len(vector), vector)

512 [-0.032297562807798386, -0.02596081607043743, 0.009842551313340664, 0.11223877966403961, -0.010131166316568851, -0.052960604429244995, -0.03155401721596718, 0.024422457441687584, 0.014341725967824459, -0.03304150328040123, 0.08955841511487961, -0.3247987926006317, -0.05275632068514824, 0.028969096019864082, -0.05952034890651703, 0.04940585419535637, 0.01561416033655405, -0.017632704228162766, -0.057727567851543427, -0.005608544684946537, 0.027480602264404297, -0.08459214121103287, -0.028209608048200607, 0.020621012896299362, 0.020411061123013496, -0.01640170067548752, -0.013907629065215588, -0.009624325670301914, -0.027314933016896248, -0.013629312627017498, 0.029618874192237854, 0.045470189303159714, -0.0194682776927948, 0.013893980532884598, -0.02371049113571644, -0.022241704165935516, 0.03953279182314873, 0.031638480722904205, -0.00932813435792923, -0.036618974059820175, 0.043379269540309906, 0.035562142729759216, 0.004908421076834202, 0.028913071379065514, -0.032781053334474564

# Milvus测试



In [8]:
from langchain_core.documents import Document
from langchain_milvus import Milvus

docs = [
    Document(page_content="LangChain 是一个用于构建基于大语言模型应用的开发框架。"),
    Document(
        page_content="Milvus 是一个开源的向量数据库，非常适合存储和检索海量向量。"
    ),
    Document(page_content="今天天气真好，我想去公园散步。"),
]

# 2. 连接 Milvus 并存入数据
print("正在连接 Milvus 并写入数据...")
vectorstore = Milvus.from_documents(
    docs,
    embedding=model,  # 使用你之前加载的 bge-small-zh-v1.5
    connection_args={"uri": "http://127.0.0.1:19530"},
    collection_name="rag_basic_test",
    drop_old=True,
)

正在连接 Milvus 并写入数据...


自动创建了数据库schema，并插入了数据

![](./images/milvus-data.png)

In [9]:
query = "什么是向量数据库？"

# 搜索最相关的 2 条结果
results = vectorstore.similarity_search(query, k=2)

In [11]:
results

[Document(metadata={'pk': 469868593122144367}, page_content='Milvus 是一个开源的向量数据库，非常适合存储和检索海量向量。'),
 Document(metadata={'pk': 469868593122144366}, page_content='LangChain 是一个用于构建基于大语言模型应用的开发框架。')]

In [ ]:
print("\n🔍 检索到的相关结果:")
for i, doc in enumerate(results):
    print(f"[{i + 1}] {doc.page_content}")


🔍 检索到的相关结果:
[1] Milvus 是一个开源的向量数据库，非常适合存储和检索海量向量。
[2] LangChain 是一个用于构建基于大语言模型应用的开发框架。


In [12]:
# 换成带分数的方法
results_with_score = vectorstore.similarity_search_with_score(query, k=2)

In [13]:
results_with_score

[(Document(metadata={'pk': 469868593122144367}, page_content='Milvus 是一个开源的向量数据库，非常适合存储和检索海量向量。'),
  0.5219870805740356),
 (Document(metadata={'pk': 469868593122144366}, page_content='LangChain 是一个用于构建基于大语言模型应用的开发框架。'),
  1.0904066562652588)]

In [ ]:
print("\n🔍 检索到的相关结果（包含得分）:")
for i, (doc, score) in enumerate(results_with_score):
    print(f"[{i + 1}] 得分: {score:.4f} | 内容: {doc.page_content}")


🔍 检索到的相关结果（包含得分）:
[1] 得分: 0.5220 | 内容: Milvus 是一个开源的向量数据库，非常适合存储和检索海量向量。
[2] 得分: 1.0904 | 内容: LangChain 是一个用于构建基于大语言模型应用的开发框架。


 L2 距离（欧氏距离）。它的规则是：数值越小，代表越相似。用 COSINE（余弦相似度）的夹角



# BAAI/bge-m3稠密向量和稀疏向量

混合检索（Hybrid Search）

现在主流的 RAG 方案（包括 BGE-M3 模型被设计出来的初衷），都是**同时生成稠密向量和稀疏向量，然后一起存入 Milvus**。
当用户提问时，系统同时发起两次检索：

1. 用稠密向量找“语义相关”的。
2. 用稀疏向量找“字面匹配”的。
   最后通过一个算法（叫 RRF 融合排序），把两边的结果综合起来。这样既能懂用户的“言外之意”，又不会漏掉核心的“关键词”。


[BAAI/bge-m3](https://hf-mirror.com/BAAI/bge-m3)

下面的代码也会走镜像和存到配置的位置，因为是从huggingface上下载，之前的配置就会生效

这个模型太大了，使用命令下载`uv run hf download BAAI/bge-m3`,支持断点续传

In [15]:
import time

from FlagEmbedding import BGEM3FlagModel

print("正在加载模型...")
start_time = time.time()
bgen3_model = BGEM3FlagModel(model_name_or_path="BAAI/bge-m3")
end_time = time.time()
print(f"模型加载完成，耗时: {end_time - start_time:.2f}秒")

正在加载模型...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

模型加载完成，耗时: 15.66秒


In [16]:
query = "我想用 LangChain 写一个 RAG 应用，怎么连接 Milvus 向量数据库？"
response = bgen3_model.encode([query], return_dense=True, return_sparse=True)

In [17]:
response

{'dense_vecs': array([[-0.02381403,  0.0154325 ,  0.00560956, ...,  0.0321954 ,
          0.06052451,  0.00811928]], shape=(1, 1024), dtype=float32),
 'lexical_weights': [defaultdict(int,
              {'1173': np.float32(0.0020972416),
               '15786': np.float32(0.17139485),
               '62467': np.float32(0.2268258),
               '73': np.float32(0.11813334),
               '9449': np.float32(0.116588905),
               '19400': np.float32(0.17250669),
               '724': np.float32(0.16382529),
               '6': np.float32(0.060737096),
               '20128': np.float32(0.16909754),
               '12082': np.float32(0.048278306),
               '82960': np.float32(0.21703133),
               '10663': np.float32(0.20604931),
               '12178': np.float32(0.30989742),
               '2110': np.float32(0.22407767),
               '3272': np.float32(0.2108431),
               '143054': np.float32(0.21934354),
               '32': np.float32(0.010007422)})],
 'co

In [20]:
lexical_weights = response["lexical_weights"]
dense_vecs = response["dense_vecs"]
sparse_vecs = bgen3_model.convert_id_to_token(lexical_weights)


In [21]:
sparse_vecs

{'用': np.float32(0.0020972416),
 'Lang': np.float32(0.17139485),
 'Cha': np.float32(0.2268258),
 'in': np.float32(0.11813334),
 '写': np.float32(0.116588905),
 'RA': np.float32(0.17250669),
 'G': np.float32(0.16382529),
 '': np.float32(0.060737096),
 '应用': np.float32(0.16909754),
 '怎么': np.float32(0.048278306),
 '连接': np.float32(0.21703133),
 'Mil': np.float32(0.20604931),
 'vus': np.float32(0.30989742),
 '向': np.float32(0.22407767),
 '量': np.float32(0.2108431),
 '数据库': np.float32(0.21934354),
 '?': np.float32(0.010007422)}